# LeetCode 549 — Binary Tree Longest Consecutive Sequence II bằng Python

Link đề: https://leetcode.com/problems/binary-tree-longest-consecutive-sequence-ii/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree, Tree DP

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def longestConsecutive(self, root: Optional[TreeNode]) -> int:
        ...
```

Ghi chú:

- Đây là bài **Premium** (cần tài khoản trả phí để xem trên LeetCode). Đề trong notebook được viết lại theo hiểu biết về bài, nên hãy **đối chiếu lại đề gốc** nếu bạn có tài khoản Premium.
- Notebook tự dựng `TreeNode` và hàm `build_tree` dựng cây từ list kiểu LeetCode (ví dụ `[2,1,3]`). Các cây tự đặt (ví dụ 3, 4 và các cây trong phần tự kiểm) đã được kiểm chứng đáp án bằng code, nhưng cũng nên **đối chiếu lại đề gốc**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **đường đi dài nhất** trong cây nhị phân mà các giá trị liên tiếp nhau **hơn kém đúng 1**, đường đi có thể **tăng** hoặc **giảm**, và có thể **uốn** qua một nút cha (dạng con → cha → con), không nhất thiết đi thẳng từ cha xuống con.
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, dựng cây từ list có `None`, đệ quy hậu tự (post-order), hàm trả về **một cặp giá trị** (tuple) và gán nhiều biến cùng lúc, `nonlocal`, bản đồ cha (`parent map`) bằng `dict`.
3. Đi từ **liệt kê mọi cặp nút rồi kiểm tra đường đi** (`O(n³)`) → **mỗi nút thử làm điểm uốn rồi tự dò hai chuỗi** (`O(n²)` ở trường hợp xấu) → **một lượt DFS hậu tự trả về cặp (inc, dec)** (`O(n)`) → **bản không đệ quy duyệt theo tầng ngược** (`O(n)`, không lo tràn ngăn xếp).
4. Hiểu mấu chốt: tại mỗi nút, chuỗi dài nhất **đi xuống theo bước +1** và chuỗi dài nhất **đi xuống theo bước −1** hợp lại thành đường đi uốn qua nút đó với độ dài `inc + dec − 1` (nút bị đếm hai lần nên trừ 1); chỉ trả **một phía** lên cho nút cha.
5. Tránh các lỗi kinh điển: ghép hai nhánh mà không kiểm tra cùng chiều (đường `1-2-1` không hợp lệ), quên trừ 1, trả luôn độ dài đã uốn lên nút cha, chỉ xét chiều tăng (nhầm sang bài 298), và để biến đếm toàn cục lưu lại giữa các lần gọi.

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một cây nhị phân. Hãy trả về **độ dài (số nút)** của **đường đi liên tiếp dài nhất** trong cây.

**Đường đi liên tiếp** là đường đi mà giá trị của hai nút kề nhau trong đường đi **hơn kém nhau đúng 1**. Đường đi này có thể **tăng dần** hoặc **giảm dần** (cả đường phải cùng một chiều).

- Ví dụ `[1,2,3,4]` và `[4,3,2,1]` đều hợp lệ; còn `[1,2,4,3]` **không** hợp lệ.
- Đường đi **có thể có dạng con → cha → con** (đi lên từ một nút rồi đi xuống nút khác), không bắt buộc đi theo chiều cha → con.

**Ví dụ 1**

- Input: `root = [1,2,3]`
- Output: `2`
- Giải thích: nút gốc có giá trị `1`, hai con có giá trị `2` và `3`. Đường dài nhất là `[1,2]` hoặc `[2,1]` (độ dài 2). Đường `2-1-3` đi qua gốc nhưng dãy giá trị `2,1,3` không đơn điệu nên không hợp lệ; còn `1` và `3` hơn kém nhau 2 nên không kề nhau trong đường liên tiếp.

**Ví dụ 2**

- Input: `root = [2,1,3]`
- Output: `3`
- Giải thích: đường dài nhất là `[1,2,3]` hoặc `[3,2,1]`, đi qua nút gốc `2` (con trái `1` → cha `2` → con phải `3`).

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = [2,1,1]`
- Output: `2`
- Giải thích: `1-2-1` đi lên rồi đi xuống nhưng giá trị `1,2,1` **không đơn điệu**, nên không hợp lệ; đường tốt nhất chỉ dài `2` (`1-2`).

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = [2,1,3,0,null,null,4]`
- Output: `5`
- Giải thích: đường `0-1-2-3-4`.

**Ràng buộc:**

- Số nút của cây nằm trong khoảng `[1, 3 * 10^4]`.
- `-3 * 10^4 <= Node.val <= 3 * 10^4`

Chú ý: cây có thể **lệch** (mỗi nút một con) với độ sâu tới `3 * 10^4`, nên đệ quy Python có thể chạm giới hạn độ sâu — ta sẽ bàn ở phần 3 và cách 4.

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def random_tree_list(rng, n, lo, hi):
    """Sinh cây ngẫu nhiên n nút (n >= 1) với giá trị trong [lo, hi], trả về list kiểu LeetCode."""
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        child = TreeNode(rng.randint(lo, hi))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return tree_to_list(root)


examples = [
    ([1, 2, 3], 2),
    ([2, 1, 3], 3),
    ([2, 1, 1], 2),                      # ví dụ tự đặt
    ([2, 1, 3, 0, None, None, 4], 5),    # ví dụ tự đặt
]
for vals, expected in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print(vals, "-> kỳ vọng", expected)

[1, 2, 3] -> kỳ vọng 2
[2, 1, 3] -> kỳ vọng 3
[2, 1, 1] -> kỳ vọng 2
[2, 1, 3, 0, None, None, 4] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Nút cây và cây nhị phân

Mỗi nút (`TreeNode`) có `val` (giá trị), `left`, `right` (nút con hoặc `None`). **Chỉ cần giữ nút gốc là truy cập được cả cây.** LeetCode định nghĩa sẵn `TreeNode` (khi nộp bài không cần viết lại), ở đây ta tự định nghĩa ở cell trên để chạy được tại máy.

In [2]:
leaf_a = TreeNode(1)
leaf_b = TreeNode(3)
root = TreeNode(2, leaf_a, leaf_b)
print(root.val, root.left.val, root.right.val)
print("con trái của lá:", leaf_a.left)        # None: lá không có con

2 1 3
con trái của lá: None


### 3.2. Dựng cây từ list kiểu LeetCode

Trong đề, cây được cho dạng `[2,1,3]`, `[2,1,3,0,None,None,4]`... theo **thứ tự từng tầng, từ trái sang phải**, `None` là chỗ trống. Hàm `build_tree` dùng một hàng đợi (`deque`): lấy nút đang chờ, gán con trái rồi con phải từ list, rồi đưa các con thật vào hàng đợi. `tree_to_list` làm ngược lại — ta dùng nó để kiểm tra việc dựng.

Hàm `show_levels` dưới đây in từng tầng để bạn "nhìn" được cây:

In [3]:
def show_levels(root):
    level, d = [root], 0
    while any(x is not None for x in level):
        print(f"tầng {d}:", [x.val if x else None for x in level])
        level = [c for x in level if x for c in (x.left, x.right)]
        d += 1

t = build_tree([2, 1, 3, 0, None, None, 4])
show_levels(t)
print(tree_to_list(t))

tầng 0: [2]
tầng 1: [1, 3]
tầng 2: [0, None, None, 4]
[2, 1, 3, 0, None, None, 4]


### 3.3. Đệ quy hậu tự (post-order)

**Post-order** là thứ tự duyệt: *xử lý hai con trước, rồi mới xử lý nút hiện tại*. Nó rất hợp khi giá trị của cha phụ thuộc vào giá trị của các con. Dưới đây ta chỉ in ra thứ tự mà các nút được "hoàn tất":

In [4]:
def postorder_values(node, out):
    if node is None:
        return
    postorder_values(node.left, out)
    postorder_values(node.right, out)
    out.append(node.val)          # chỉ ghi nhận SAU khi hai con đã xong

res = []
postorder_values(build_tree([2, 1, 3, 0, None, None, 4]), res)
print(res)       # lá trước, gốc sau cùng

[0, 1, 4, 3, 2]


### 3.4. Hàm trả về một cặp giá trị (tuple) và gán nhiều biến cùng lúc

Trong bài này, mỗi nút cần báo cho cha **hai** con số (độ dài chuỗi tăng, độ dài chuỗi giảm). Python cho phép `return a, b` (thực chất trả một **tuple**) và nhận bằng `x, y = f()`. Đây gọi là *unpacking*.

In [5]:
def two_numbers(x):
    return x + 1, x - 1        # trả về tuple (x+1, x-1)

up, down = two_numbers(10)
print(up, down)
pair = two_numbers(0)
print(pair, type(pair))
inc = dec = 1                  # gán cùng giá trị cho hai biến
print(inc, dec)

11 9
(1, -1) <class 'tuple'>
1 1


### 3.5. Hàm lồng nhau và `nonlocal`

Hàm `dfs` thường được định nghĩa **bên trong** hàm chính để dùng chung biến (như biến `ans` lưu đáp án tốt nhất). Nếu chỉ **đọc** hay **gọi phương thức** trên biến ngoài thì không cần gì thêm. Nhưng nếu muốn **gán lại** một biến số (`ans = max(...)`), phải khai báo `nonlocal ans`, nếu không Python hiểu `ans` là biến cục bộ mới và báo lỗi.

In [6]:
def demo():
    best = 0
    def update(x):
        nonlocal best          # vì có phép gán best = ...
        best = max(best, x)
    for v in [3, 9, 4]:
        update(v)
    return best

print(demo())

9


### 3.6. Đường đi trong cây: đi lên rồi đi xuống

Trong cây có `n` nút, giữa **hai nút bất kỳ** `u` và `v` luôn có **đúng một** đường đơn: đi **lên** từ `u` tới tổ tiên chung thấp nhất (LCA), rồi đi **xuống** tới `v`. Với đề này, đường `con → cha → con` chính là đường đi lên một bước rồi xuống một bước.

Để đi lên được ta cần biết cha của mỗi nút — cây chỉ lưu con. Cách quen thuộc: duyệt một lần, ghi `parent[nút] = cha` vào một `dict` (khóa là chính đối tượng nút). Ta cũng ghi độ sâu `depth` để biết bên nào phải đi lên trước.

In [7]:
def parent_map(root):
    par, depth, order = {root: None}, {root: 0}, []
    stack = [root]
    while stack:
        node = stack.pop()
        order.append(node)
        for ch in (node.left, node.right):
            if ch is not None:
                par[ch] = node
                depth[ch] = depth[node] + 1
                stack.append(ch)
    return par, depth, order

def path_between(u, v, par, depth):
    """Danh sách giá trị trên đường đơn từ u tới v (đi lên rồi đi xuống)."""
    left, right = [], []
    while depth[u] > depth[v]:
        left.append(u); u = par[u]
    while depth[v] > depth[u]:
        right.append(v); v = par[v]
    while u is not v:
        left.append(u); right.append(v)
        u = par[u]; v = par[v]
    return [x.val for x in left] + [u.val] + [x.val for x in reversed(right)]

t = build_tree([2, 1, 3, 0, None, None, 4])
par, depth, order = parent_map(t)
by_val = {x.val: x for x in order}          # ở cây này các giá trị khác nhau
print(path_between(by_val[0], by_val[4], par, depth))   # 0-1-2-3-4
print(path_between(by_val[1], by_val[3], par, depth))   # 1-2-3
print(path_between(by_val[4], by_val[4], par, depth))   # chỉ một nút

[0, 1, 2, 3, 4]
[1, 2, 3]
[4]


### 3.7. Kiểm tra "dãy liên tiếp đơn điệu" và độ sâu đệ quy

Một dãy giá trị `vals` là **liên tiếp** khi hiệu hai phần tử kề nhau luôn bằng **cùng một** số `d` và `d` chỉ có thể là `+1` hoặc `-1` (dãy một phần tử luôn hợp lệ). Hàm `is_consecutive` bên dưới sẽ được dùng làm "trọng tài" trong các test.

Ngoài ra, Python mặc định chỉ cho đệ quy sâu khoảng `1000` tầng (`sys.getrecursionlimit()`). Cây lệch `3 * 10^4` nút cần nới giới hạn bằng `sys.setrecursionlimit(...)` hoặc dùng bản không đệ quy.

In [8]:
def is_consecutive(vals):
    if len(vals) <= 1:
        return True
    d = vals[1] - vals[0]
    return d in (1, -1) and all(vals[i + 1] - vals[i] == d for i in range(len(vals) - 1))

print(is_consecutive([1, 2, 3, 4]), is_consecutive([4, 3, 2, 1]))
print(is_consecutive([1, 2, 4, 3]), is_consecutive([1, 2, 1]), is_consecutive([5]))
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

True True
False False True
giới hạn đệ quy mặc định: 1000


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét cây `[2, 1, 3, 0, None, None, 4]`:

```
        2
       / \
      1   3
     /     \
    0       4
```

Ta thấy ngay đường `0 - 1 - 2 - 3 - 4` có độ dài `5`: nó **đi lên** từ `0` qua `1` tới `2` (giá trị tăng dần), rồi **đi xuống** qua `3` tới `4` (vẫn tăng dần).

Hãy nhìn tại nút `2` (nút "uốn"):

- Về phía **con trái**: từ `2` đi xuống ta gặp `1`, rồi `0` → giá trị giảm `2, 1, 0`. Đây là chuỗi **giảm** đi xuống, dài `3` nút (tính cả `2`).
- Về phía **con phải**: từ `2` đi xuống ta gặp `3`, rồi `4` → giá trị tăng `2, 3, 4`. Chuỗi **tăng** đi xuống, dài `3` nút (tính cả `2`).

Ghép hai chuỗi tại `2`: `3 + 3 − 1 = 5` (nút `2` bị đếm hai lần nên phải trừ `1`).

Đây chính là điểm mấu chốt:

> Một đường đi liên tiếp dài nhất, nhìn từ **nút cao nhất** (gần gốc nhất) của nó, luôn gồm **tối đa hai chuỗi đi xuống**: một chuỗi theo bước `+1` và một chuỗi theo bước `−1`, xuất phát từ nút đó và đi vào **hai con khác nhau**.

Vì sao hai chuỗi phải khác chiều bước? Ghép `2, 3, 4` (bước +1 khi đi xuống) với `2, 1, 0` (bước −1) rồi đọc chuỗi thứ hai ngược lại cho ra `0, 1, 2, 3, 4`: đơn điệu. Nếu cả hai chuỗi cùng bước +1 thì dãy ghép sẽ là `4, 3, 2, 2, 3, 4`-kiểu "gập" (xuống rồi lên), không đơn điệu.

In [9]:
# Tính tay bằng code: chuỗi đi xuống từ nút 2 theo từng phía
t = build_tree([2, 1, 3, 0, None, None, 4])
left_chain, node = [t.val], t.left
while node:
    left_chain.append(node.val); node = node.left
right_chain, node = [t.val], t.right
while node:
    right_chain.append(node.val); node = node.right
print("trái :", left_chain)        # [2, 1, 0]  -> giảm
print("phải :", right_chain)       # [2, 3, 4]  -> tăng
print("độ dài đường uốn:", len(left_chain) + len(right_chain) - 1)

trái : [2, 1, 0]
phải : [2, 3, 4]
độ dài đường uốn: 5


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi cặp nút và kiểm tra đường đi

**Ý tưởng "dịch nguyên văn đề bài":**

1. Mỗi đường đi trong cây được xác định bởi **hai đầu mút** `(u, v)` (đường đơn duy nhất giữa chúng).
2. Với mọi cặp `(u, v)` (kể cả `u = v`), dựng danh sách giá trị trên đường đi bằng `path_between` ở mục 3.6.
3. Nếu danh sách đó là **liên tiếp** (`is_consecutive`) thì cập nhật đáp án bằng độ dài của nó.

Đây là cách chắc chắn đúng, và ta sẽ dùng nó làm **oracle** (đáp án đối chứng độc lập) cho các cách nhanh hơn.

In [10]:
def longest_pairs(root):
    par, depth, order = parent_map(root)
    best = 1
    for i in range(len(order)):
        for j in range(i, len(order)):
            vals = path_between(order[i], order[j], par, depth)
            if is_consecutive(vals):
                best = max(best, len(vals))
    return best

for vals, expected in examples:
    got = longest_pairs(build_tree(vals))
    print(vals, "->", got, "| kỳ vọng", expected)
    assert got == expected

[1, 2, 3] -> 2 | kỳ vọng 2
[2, 1, 3] -> 3 | kỳ vọng 3
[2, 1, 1] -> 2 | kỳ vọng 2
[2, 1, 3, 0, None, None, 4] -> 5 | kỳ vọng 5


**Độ phức tạp.** Có `n(n+1)/2` cặp; mỗi cặp dựng đường đi dài tới `O(n)` rồi kiểm tra.

- **Time:** `O(n³)` trường hợp xấu (cây lệch). Với `n = 3 * 10^4` là không thể chạy.
- **Space:** `O(n)` cho `par`, `depth`, `order` và danh sách đường đi.

**Nút thắt:** ta dựng và kiểm tra cả những đường **hỏng ngay từ cạnh đầu tiên** (ví dụ hai nút kề nhau đã hơn kém `7`), và dựng lại đường đi từ đầu cho từng cặp. Cần một cách nhìn **theo từng nút làm điểm uốn** thay vì theo từng cặp.

## 6. Ý tưởng thuật toán / bước đột phá

**Nhóm đường đi theo nút cao nhất.** Mỗi đường đi có đúng **một** nút cao nhất (gần gốc nhất) gọi là *đỉnh* `v`. Đường đi đó = (một chuỗi đi xuống từ `v` vào con này) + `v` + (một chuỗi đi xuống từ `v` vào con kia). Nếu đường là liên tiếp và đơn điệu thì hai chuỗi này **ngược chiều bước**: một chuỗi theo `+1`, chuỗi kia theo `−1` (khi nhìn từ `v` đi xuống).

Định nghĩa với mỗi nút `v`:

- `inc[v]` = độ dài chuỗi **dài nhất đi xuống** bắt đầu tại `v` với mỗi bước **+1** (`v, v+1, v+2, ...`).
- `dec[v]` = độ dài chuỗi dài nhất đi xuống bắt đầu tại `v` với mỗi bước **−1** (`v, v−1, v−2, ...`).

Công thức:

```
inc[v] = 1 + max(inc[c])  trên các con c có c.val == v.val + 1   (không có con như vậy thì inc[v] = 1)
dec[v] = 1 + max(dec[c])  trên các con c có c.val == v.val - 1   (không có thì dec[v] = 1)
đường uốn tốt nhất có đỉnh v:  inc[v] + dec[v] - 1
```

Hai chuỗi không thể cùng đi qua **một** con `c`, vì `c.val` không thể vừa bằng `v.val + 1` vừa bằng `v.val − 1`. Do đó khi cộng `inc[v] + dec[v] − 1` ta **tự động** có hai nhánh khác nhau (hoặc một trong hai chuỗi chỉ là chính `v`). Đáp án là `max` của biểu thức đó trên mọi nút `v`.

Ta thử công thức bằng tay trên hàm đệ quy "chuỗi đi xuống" đơn giản trước khi tối ưu:

In [11]:
def chain(node, need, d):
    """Độ dài chuỗi đi xuống: nút đầu phải có giá trị `need`, mỗi bước tiếp theo cộng d."""
    if node is None or node.val != need:
        return 0
    return 1 + max(chain(node.left, need + d, d), chain(node.right, need + d, d))

t = build_tree([2, 1, 3, 0, None, None, 4])
print("chuỗi +1 từ nút 3 :", chain(t.right, 3, +1))   # 3, 4 -> 2
print("chuỗi -1 từ nút 1 :", chain(t.left, 1, -1))    # 1, 0 -> 2
print("đường uốn qua gốc :", 1 + chain(t.right, t.val + 1, +1) + chain(t.left, t.val - 1, -1))

chuỗi +1 từ nút 3 : 2
chuỗi -1 từ nút 1 : 2
đường uốn qua gốc : 5


## 7. Các cách tiếp theo

### Cách 2 — Mỗi nút làm đỉnh, tự dò hai chuỗi (`O(n²)` xấu nhất)

**Giải thích từng đoạn:**

1. Duyệt qua **mọi nút** `v` làm đỉnh.
2. Với mỗi `v`, thử hai cách phân công hai con: (con trái đi theo `+1`, con phải đi theo `−1`) và ngược lại. Độ dài đường = `1 + chain(a, v+1, +1) + chain(b, v-1, -1)`.
3. Chú ý: hàm `chain` được gọi lại **từ đầu** cho từng đỉnh, nên một chuỗi dài bị dò nhiều lần.

Vì sao chỉ cần hai tổ hợp? Nhìn từ đỉnh `v`, một con nhận chuỗi bước `+1` (giá trị `v+1, v+2, ...`), con còn lại nhận chuỗi bước `−1` (giá trị `v−1, v−2, ...`). Chỉ có hai cách phân công: (trái `+1`, phải `−1`) hoặc (phải `+1`, trái `−1`). Nếu con nào không khớp giá trị thì `chain` trả `0`, nên trường hợp "chuỗi chỉ có `v`" hay "chỉ đi một phía" cũng được tính.

In [12]:
def longest_each_node(root):
    best = 0
    stack = [root]
    while stack:
        v = stack.pop()
        if v is None:
            continue
        for a, b in ((v.left, v.right), (v.right, v.left)):
            best = max(best, 1 + chain(a, v.val + 1, +1) + chain(b, v.val - 1, -1))
        stack.append(v.left)
        stack.append(v.right)
    return best

for vals, expected in examples:
    got = longest_each_node(build_tree(vals))
    print(vals, "->", got)
    assert got == expected

[1, 2, 3] -> 2
[2, 1, 3] -> 3
[2, 1, 1] -> 2
[2, 1, 3, 0, None, None, 4] -> 5


In [13]:
class Solution:
    def longestConsecutive(self, root: Optional[TreeNode]) -> int:
        def chain(node, need, d):
            if node is None or node.val != need:
                return 0
            return 1 + max(chain(node.left, need + d, d), chain(node.right, need + d, d))

        best = 0
        stack = [root]
        while stack:
            v = stack.pop()
            if v is None:
                continue
            for a, b in ((v.left, v.right), (v.right, v.left)):
                best = max(best, 1 + chain(a, v.val + 1, 1) + chain(b, v.val - 1, -1))
            stack.append(v.left)
            stack.append(v.right)
        return best

print([Solution().longestConsecutive(build_tree(v)) for v, _ in examples])

[2, 3, 2, 5]


**Độ phức tạp (cách 2):**

- **Time:** `O(n²)` trường hợp xấu. Ví dụ cây lệch có giá trị `1, 2, 3, ..., n`: từ đỉnh thứ `i` ta dò chuỗi dài `n − i` → tổng `~ n²/2`. Cây thường gặp nhanh hơn nhiều, nhưng không có đảm bảo.
- **Space:** `O(h)` ngăn xếp đệ quy của `chain` cộng ngăn xếp duyệt `O(n)` ở trường hợp xấu.

**Nút thắt còn lại:** `chain(a, ...)` tính lại cùng một chuỗi con nhiều lần (khi tính cho `v` rồi lại khi tính cho cha của `v`). Hãy để mỗi nút **tính độ dài chuỗi của chính nó đúng một lần** và báo lên cha.

### Cách 3 — DFS hậu tự trả về cặp `(inc, dec)` (cách chuẩn, `O(n)`)

Đây là công thức ở mục 6 viết thẳng thành đệ quy.

**Giải thích từng đoạn:**

1. `dfs(node)` trả về `(inc, dec)` của `node`. Gặp `None` trả `(0, 0)` (không có nút, không có chuỗi).
2. Khởi tạo `inc = dec = 1` (chính `node` là một chuỗi dài 1).
3. Với mỗi con `ch`: gọi `ci, cd = dfs(ch)` (**phải gọi cho cả hai con**). Nếu `ch.val == node.val + 1` thì chuỗi tăng kéo dài: `inc = max(inc, ci + 1)`. Nếu `ch.val == node.val - 1` thì `dec = max(dec, cd + 1)`.
4. Ghi nhận đường uốn qua `node`: `ans = max(ans, inc + dec - 1)`.
5. Trả `(inc, dec)` lên cha. **Chỉ trả một phía**, không trả `inc + dec - 1`, vì nếu cha lấy đường đã uốn thì đường đi sẽ có hai ngã rẽ (không còn là đường đơn).

In [14]:
def longest_dp(root):
    ans = 0

    def dfs(node):
        nonlocal ans
        if node is None:
            return 0, 0
        inc = dec = 1
        for ch in (node.left, node.right):
            if ch is None:
                continue
            ci, cd = dfs(ch)
            if ch.val == node.val + 1:
                inc = max(inc, ci + 1)
            elif ch.val == node.val - 1:
                dec = max(dec, cd + 1)
        ans = max(ans, inc + dec - 1)
        return inc, dec

    dfs(root)
    return ans

for vals, expected in examples:
    got = longest_dp(build_tree(vals))
    print(vals, "->", got)
    assert got == expected

[1, 2, 3] -> 2
[2, 1, 3] -> 3
[2, 1, 1] -> 2
[2, 1, 3, 0, None, None, 4] -> 5


In [15]:
class Solution:
    def longestConsecutive(self, root: Optional[TreeNode]) -> int:
        ans = 0

        def dfs(node: Optional[TreeNode]) -> tuple:
            nonlocal ans
            if node is None:
                return 0, 0
            inc = dec = 1
            for ch in (node.left, node.right):
                if ch is None:
                    continue
                ci, cd = dfs(ch)
                if ch.val == node.val + 1:
                    inc = max(inc, ci + 1)
                elif ch.val == node.val - 1:
                    dec = max(dec, cd + 1)
            ans = max(ans, inc + dec - 1)
            return inc, dec

        dfs(root)
        return ans

print([Solution().longestConsecutive(build_tree(v)) for v, _ in examples])

[2, 3, 2, 5]


**Độ phức tạp (cách 3):**

- **Time:** `O(n)` — mỗi nút được thăm đúng một lần, xử lý `O(1)` (hai con).
- **Space:** `O(h)` ngăn xếp đệ quy, `h` là chiều cao cây (tới `n` với cây lệch).

### Cách 4 — Không đệ quy: duyệt theo tầng rồi xử lý **ngược** (`O(n)`, an toàn với cây lệch)

Cây lệch `3 * 10^4` nút có thể làm đệ quy quá sâu trên máy cá nhân (và đôi khi trên judge). Ta tránh đệ quy bằng quan sát: duyệt theo tầng (BFS) thì **con luôn nằm sau cha** trong danh sách duyệt. Do đó **đọc danh sách ngược** (từ cuối về đầu) là một thứ tự hợp lệ để tính `inc, dec`: khi gặp một nút, hai con của nó đã có kết quả.

**Giải thích từng đoạn:**

1. `order = [root]`, mở rộng bằng hàng đợi (dùng chỉ số `i` chạy dọc `order`, không cần `popleft`).
2. `inc` và `dec` là hai `dict` từ **nút** sang giá trị.
3. `for v in reversed(order)`: tính `a, b` (inc, dec của `v`) theo cùng công thức, ghi `inc[v] = a`, `dec[v] = b`, cập nhật `ans`.

In [16]:
def longest_iter(root):
    order = [root]
    i = 0
    while i < len(order):
        v = order[i]
        i += 1
        for ch in (v.left, v.right):
            if ch is not None:
                order.append(ch)

    inc, dec, ans = {}, {}, 0
    for v in reversed(order):            # con luôn được xử lý trước cha
        a = b = 1
        for ch in (v.left, v.right):
            if ch is None:
                continue
            if ch.val == v.val + 1:
                a = max(a, inc[ch] + 1)
            elif ch.val == v.val - 1:
                b = max(b, dec[ch] + 1)
        inc[v], dec[v] = a, b
        ans = max(ans, a + b - 1)
    return ans

for vals, expected in examples:
    got = longest_iter(build_tree(vals))
    print(vals, "->", got)
    assert got == expected

[1, 2, 3] -> 2
[2, 1, 3] -> 3
[2, 1, 1] -> 2
[2, 1, 3, 0, None, None, 4] -> 5


In [17]:
class Solution:
    def longestConsecutive(self, root: Optional[TreeNode]) -> int:
        order = [root]
        i = 0
        while i < len(order):
            v = order[i]
            i += 1
            for ch in (v.left, v.right):
                if ch is not None:
                    order.append(ch)

        inc, dec, ans = {}, {}, 0
        for v in reversed(order):
            a = b = 1
            for ch in (v.left, v.right):
                if ch is None:
                    continue
                if ch.val == v.val + 1:
                    a = max(a, inc[ch] + 1)
                elif ch.val == v.val - 1:
                    b = max(b, dec[ch] + 1)
            inc[v], dec[v] = a, b
            ans = max(ans, a + b - 1)
        return ans

print([Solution().longestConsecutive(build_tree(v)) for v, _ in examples])

[2, 3, 2, 5]


**Độ phức tạp (cách 4):**

- **Time:** `O(n)` — hai lượt qua danh sách `order`.
- **Space:** `O(n)` cho `order`, `inc`, `dec` (không còn ngăn xếp đệ quy sâu).

## 8. TRACE từng bước

Ta trace **cách 3** trên cây `[2, 1, 3, 0, None, None, 4]`. Hậu tự nên thứ tự "hoàn tất" nút là `0, 1, 4, 3, 2`.

| Bước | Nút `v` | Con và giá trị liên quan | `inc` | `dec` | `inc+dec−1` | `ans` |
|---|---|---|---|---|---|---|
| 1 | `0` (lá) | không có con | 1 | 1 | 1 | 1 |
| 2 | `1` | con trái `0 = 1 − 1` → `dec = dec[0] + 1` | 1 | 2 | 2 | 2 |
| 3 | `4` (lá) | không có con | 1 | 1 | 1 | 2 |
| 4 | `3` | con phải `4 = 3 + 1` → `inc = inc[4] + 1` | 2 | 1 | 2 | 2 |
| 5 | `2` (gốc) | trái `1 = 2 − 1` → `dec = dec[1] + 1 = 3`; phải `3 = 2 + 1` → `inc = inc[3] + 1 = 3` | 3 | 3 | **5** | **5** |

Hàm `trace_dp` dưới đây in đúng các bước đó, kèm lý do từng con có/không kéo dài chuỗi.

In [18]:
def trace_dp(root):
    ans = 0
    step = 0

    def dfs(node, depth):
        nonlocal ans, step
        if node is None:
            return 0, 0
        inc = dec = 1
        notes = []
        for name, ch in (("trái", node.left), ("phải", node.right)):
            if ch is None:
                continue
            ci, cd = dfs(ch, depth + 1)
            if ch.val == node.val + 1:
                inc = max(inc, ci + 1)
                notes.append(f"con {name} {ch.val} = {node.val}+1 -> inc = {ci}+1")
            elif ch.val == node.val - 1:
                dec = max(dec, cd + 1)
                notes.append(f"con {name} {ch.val} = {node.val}-1 -> dec = {cd}+1")
            else:
                notes.append(f"con {name} {ch.val}: lệch bước, không nối")
        ans = max(ans, inc + dec - 1)
        step += 1
        print(f"bước {step}: nút {node.val:>2} | inc={inc} dec={dec} | inc+dec-1={inc + dec - 1} | ans={ans}"
              + ("" if not notes else "\n         " + "; ".join(notes)))
        return inc, dec

    dfs(root, 0)
    return ans

print("kết quả:", trace_dp(build_tree([2, 1, 3, 0, None, None, 4])))

bước 1: nút  0 | inc=1 dec=1 | inc+dec-1=1 | ans=1
bước 2: nút  1 | inc=1 dec=2 | inc+dec-1=2 | ans=2
         con trái 0 = 1-1 -> dec = 1+1
bước 3: nút  4 | inc=1 dec=1 | inc+dec-1=1 | ans=2
bước 4: nút  3 | inc=2 dec=1 | inc+dec-1=2 | ans=2
         con phải 4 = 3+1 -> inc = 1+1
bước 5: nút  2 | inc=3 dec=3 | inc+dec-1=5 | ans=5
         con trái 1 = 2-1 -> dec = 2+1; con phải 3 = 2+1 -> inc = 2+1
kết quả: 5


Trace thứ hai trên một cây có **nhánh nhiễu** (giá trị không liên tiếp) để thấy các nhánh bị bỏ qua: `[5, 4, 6, 3, 9, 7, 3]`.

```
          5
        /   \
       4     6
      / \   / \
     3   9 7   3
```

Nút `5` có hai con `4` (giảm) và `6` (tăng). Con `4` có con `3` (giảm tiếp) và `9` (nhiễu). Con `6` có con `7` (tăng tiếp) và `3` (nhiễu). Kỳ vọng: đường `3-4-5-6-7` dài `5`.

In [19]:
print("kết quả:", trace_dp(build_tree([5, 4, 6, 3, 9, 7, 3])))

bước 1: nút  3 | inc=1 dec=1 | inc+dec-1=1 | ans=1
bước 2: nút  9 | inc=1 dec=1 | inc+dec-1=1 | ans=1
bước 3: nút  4 | inc=1 dec=2 | inc+dec-1=2 | ans=2
         con trái 3 = 4-1 -> dec = 1+1; con phải 9: lệch bước, không nối
bước 4: nút  7 | inc=1 dec=1 | inc+dec-1=1 | ans=2
bước 5: nút  3 | inc=1 dec=1 | inc+dec-1=1 | ans=2
bước 6: nút  6 | inc=2 dec=1 | inc+dec-1=2 | ans=2
         con trái 7 = 6+1 -> inc = 1+1; con phải 3: lệch bước, không nối
bước 7: nút  5 | inc=3 dec=3 | inc+dec-1=5 | ans=5
         con trái 4 = 5-1 -> dec = 2+1; con phải 6 = 5+1 -> inc = 2+1
kết quả: 5


## 9. Code nộp LeetCode

Đây là cách 3 (chuẩn). Khi nộp lên LeetCode **không cần** định nghĩa `TreeNode`, và `Optional` đã được import sẵn; ở đây ta giữ `from typing import Optional` ở đầu notebook để chạy cục bộ.

Nếu bạn lo cây lệch quá sâu làm tràn ngăn xếp đệ quy, hãy nộp **cách 4** (không đệ quy) ở trên.

In [20]:
class Solution:
    def longestConsecutive(self, root: Optional[TreeNode]) -> int:
        ans = 0

        def dfs(node: Optional[TreeNode]) -> tuple:
            nonlocal ans
            if node is None:
                return 0, 0
            inc = dec = 1
            for ch in (node.left, node.right):
                if ch is None:
                    continue
                ci, cd = dfs(ch)
                if ch.val == node.val + 1:
                    inc = max(inc, ci + 1)
                elif ch.val == node.val - 1:
                    dec = max(dec, cd + 1)
            ans = max(ans, inc + dec - 1)
            return inc, dec

        dfs(root)
        return ans


sol = Solution()
for vals, expected in examples:
    got = sol.longestConsecutive(build_tree(vals))
    print(vals, "->", got)
    assert got == expected
print("Chạy ví dụ: OK")

[1, 2, 3] -> 2
[2, 1, 3] -> 3
[2, 1, 1] -> 2
[2, 1, 3, 0, None, None, 4] -> 5
Chạy ví dụ: OK


## 10. Vì sao đúng

**Mệnh đề 1 (phân rã đường đi).** Mọi đường đi liên tiếp `P` có đúng một nút cao nhất `v`. Bỏ `v` ra thì `P` tách thành tối đa hai đoạn, mỗi đoạn nằm trong cây con của **một** con khác nhau của `v`, và mỗi đoạn đi **xuống** từ `v`. Do `P` đơn điệu nên nếu một đoạn có bước `+1` (khi đi từ `v` xuống) thì đoạn kia, vì nối tiếp ngược chiều qua `v`, phải có bước `−1`. Vậy `P` = (chuỗi `+1` xuống từ `v`) ∪ (chuỗi `−1` xuống từ `v`) và `|P| = len₊ + len₋ − 1`.

**Mệnh đề 2 (định nghĩa đệ quy).** `inc[v]` là độ dài chuỗi `+1` dài nhất đi xuống từ `v`: chuỗi chỉ gồm `v` (độ dài 1) hoặc đi tiếp vào con `c` có `c.val = v.val + 1` rồi theo chuỗi `+1` dài nhất từ `c`. Vậy `inc[v] = 1 + max(0, inc[c] cho các con c thoả mãn)`; tương tự `dec`.

**Mệnh đề 3 (hai chuỗi không dùng chung con).** Một con `c` không thể vừa có `c.val = v.val + 1` vừa có `c.val = v.val − 1`, nên chuỗi `+1` và chuỗi `−1` đi vào hai con khác nhau (hoặc chỉ là chính `v`). Do đó `inc[v] + dec[v] − 1` là độ dài của một **đường đơn thật** và nó là đường tốt nhất có đỉnh `v`.

**Kết luận.** Lấy `max` trên mọi `v` cho ra đường đi dài nhất. Trả về cho cha chỉ `inc[v]` và `dec[v]` (không phải đường đã uốn) là đúng, vì đường đi qua cả `cha`, `v` và đi tiếp xuống con của `v` chỉ có thể là một chuỗi đơn hướng bắt đầu tại `v`.

Ta kiểm chứng định nghĩa bằng code: so `inc, dec` của mọi nút do DP tính với `chain` tính trực tiếp từ định nghĩa, trên nhiều cây ngẫu nhiên.

In [21]:
def all_inc_dec(root):
    """DP trả về dict nút -> (inc, dec) cho mọi nút."""
    res = {}
    def dfs(node):
        if node is None:
            return 0, 0
        inc = dec = 1
        for ch in (node.left, node.right):
            if ch is None:
                continue
            ci, cd = dfs(ch)
            if ch.val == node.val + 1:
                inc = max(inc, ci + 1)
            elif ch.val == node.val - 1:
                dec = max(dec, cd + 1)
        res[node] = (inc, dec)
        return inc, dec
    dfs(root)
    return res

rng = random.Random(2024)
checked = 0
for _ in range(300):
    n = rng.randint(1, 12)
    root = build_tree(random_tree_list(rng, n, 1, 4))
    for node, (inc, dec) in all_inc_dec(root).items():
        assert inc == chain(node, node.val, +1)      # định nghĩa trực tiếp
        assert dec == chain(node, node.val, -1)
        checked += 1
print("đã kiểm chứng inc/dec trên", checked, "nút")

đã kiểm chứng inc/dec trên 1919 nút


## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi cặp nút | `O(n³)` | `O(n)` | chỉ để đối chứng |
| 2. Mỗi nút dò chuỗi | `O(n²)` xấu nhất | `O(n)` | tính lại chuỗi nhiều lần |
| 3. DFS hậu tự `(inc, dec)` | `O(n)` | `O(h)` | cách chuẩn |
| 4. BFS ngược, không đệ quy | `O(n)` | `O(n)` | an toàn với cây lệch |

Với cách 3: mỗi nút được gọi `dfs` đúng một lần và làm công việc hằng số (hai con) nên Time `O(n)`. Ngăn xếp đệ quy sâu bằng chiều cao `h` (`O(log n)` với cây cân bằng, `O(n)` với cây lệch), nên Space `O(h)`.

Cách 4 đổi ngăn xếp đệ quy lấy hai `dict` và danh sách `order`, Space `O(n)`, nhưng không bao giờ gặp `RecursionError`.

## 12. Bộ test

Gồm: ví dụ đề, edge case (cây một nút, toàn giá trị bằng nhau, chuỗi tăng/giảm thuần, zigzag), phản ví dụ cho cách làm sai, đối chiếu bốn cách trên nhiều cây ngẫu nhiên (có seed) và test lớn đúng ràng buộc.

In [22]:
ALL_METHODS = {
    "pairs": longest_pairs,
    "each_node": longest_each_node,
    "dp": longest_dp,
    "iter": longest_iter,
}

def check_all(vals, expected, methods=ALL_METHODS):
    for name, fn in methods.items():
        got = fn(build_tree(vals))
        assert got == expected, (name, vals, got, expected)

# ví dụ đề + tự đặt
for vals, expected in examples:
    check_all(vals, expected)

# edge case
check_all([1], 1)                                  # cây một nút
check_all([5, 5, 5], 1)                            # toàn giá trị bằng nhau
check_all([1, 2, None, 3, None, 4], 4)             # chuỗi tăng thuần, cây lệch trái
check_all([4, None, 3, None, 2, None, 1], 4)       # chuỗi giảm thuần, cây lệch phải
check_all([1, 3, None, 5], 1)                      # các cạnh đều hơn kém 2: không nối được gì
check_all([1, 3, None, 2], 2)                      # chỉ có cặp 3-2
check_all([2, 1, 1], 2)                            # 1-2-1 không đơn điệu
check_all([2, 3, 1], 3)                            # 3-2-1 đi qua gốc
check_all([-1, 0, -2], 3)                          # giá trị âm: -2 -1 0
check_all([0, 1, -1, 2, None, None, -2], 5)        # -2 -1 0 1 2
check_all([3, 2, 4, 1, None, None, 5], 5)          # 1-2-3-4-5
check_all([1, 2, 3, 3, 4, 4, 5], 3)                # nhiều chuỗi, có nhánh nhiễu
print("edge case: OK")

edge case: OK


**Phản ví dụ** cho hai cách làm sai điển hình (chi tiết ở mục 13):

- Ghép hai nhánh **không** kiểm tra chiều: cây `[2, 1, 1]` bị đếm thành `3` thay vì `2`.
- Chỉ xét chuỗi **tăng** đi xuống (bài 298): cây `[4, None, 3, None, 2, None, 1]` bị đếm `1` thay vì `4`.

In [23]:
def wrong_merge(root):
    """SAI: cộng hai nhánh bất kể chiều."""
    ans = 0
    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        l = dfs(node.left)
        r = dfs(node.right)
        a = l + 1 if node.left and abs(node.left.val - node.val) == 1 else 1
        b = r + 1 if node.right and abs(node.right.val - node.val) == 1 else 1
        ans = max(ans, a + b - 1)
        return max(a, b)
    dfs(root)
    return ans

print(wrong_merge(build_tree([2, 1, 1])), "(đúng phải là 2)")
assert wrong_merge(build_tree([2, 1, 1])) != 2

3 (đúng phải là 2)


In [24]:
# Đối chiếu 4 cách trên nhiều cây ngẫu nhiên nhỏ (seed cố định), giá trị trong khoảng hẹp để có nhiều chuỗi
rng = random.Random(549)
count = 0
for _ in range(1500):
    n = rng.randint(1, 10)
    lo, hi = rng.choice([(1, 3), (1, 4), (-2, 2), (0, 6)])
    vals = random_tree_list(rng, n, lo, hi)
    exp = longest_pairs(build_tree(vals))
    for name, fn in ALL_METHODS.items():
        assert fn(build_tree(vals)) == exp, (name, vals)
    count += 1
print("đã đối chiếu", count, "cây ngẫu nhiên nhỏ: OK")

# cây lớn hơn: bỏ cách 1 (quá chậm), so cách 2, 3, 4 với nhau
fast = {k: ALL_METHODS[k] for k in ("each_node", "dp", "iter")}
for _ in range(200):
    n = rng.randint(30, 200)
    vals = random_tree_list(rng, n, 1, 4)
    res = {name: fn(build_tree(vals)) for name, fn in fast.items()}
    assert len(set(res.values())) == 1, (vals, res)
print("đối chiếu cây 30..200 nút: OK")

đã đối chiếu 1500 cây ngẫu nhiên nhỏ: OK


đối chiếu cây 30..200 nút: OK


In [25]:
# Test lớn đúng ràng buộc: cây lệch 30000 nút (cần nới giới hạn đệ quy cho cách 3)
sys.setrecursionlimit(200000)
N = 30000

def skew_chain(values):
    """Cây lệch trái: values[0] là gốc, mỗi nút có đúng một con trái."""
    root = TreeNode(values[0])
    cur = root
    for x in values[1:]:
        cur.left = TreeNode(x)
        cur = cur.left
    return root

chain_up = skew_chain(list(range(1, N + 1)))          # 1,2,...,30000 -> đáp án 30000
chain_zig = skew_chain([1 + (i % 2) for i in range(N)])  # 1,2,1,2,... -> đáp án 2
chain_dec = skew_chain(list(range(N, 0, -1)))
for name, root, exp in (("tăng", chain_up, N), ("zigzag", chain_zig, 2), ("giảm", chain_dec, N)):
    t0 = time.perf_counter()
    r3 = longest_dp(root)
    t3 = time.perf_counter() - t0
    t0 = time.perf_counter()
    r4 = longest_iter(root)
    t4 = time.perf_counter() - t0
    assert r3 == r4 == exp, (name, r3, r4, exp)
    print(f"cây lệch {name}: đáp án {r3}  (DFS {t3*1000:.0f} ms, BFS ngược {t4*1000:.0f} ms)")

# cây ngẫu nhiên 30000 nút, giá trị trong [-3*10^4, 3*10^4]
rng = random.Random(30000)
vals = random_tree_list(rng, N, -30000, 30000)
r3, r4 = longest_dp(build_tree(vals)), longest_iter(build_tree(vals))
assert r3 == r4
print("cây ngẫu nhiên 30000 nút, giá trị rộng:", r3, r4)

# cây ngẫu nhiên 30000 nút, giá trị hẹp để có chuỗi dài
vals = random_tree_list(rng, N, 1, 3)
r3, r4 = longest_dp(build_tree(vals)), longest_iter(build_tree(vals))
assert r3 == r4
print("cây ngẫu nhiên 30000 nút, giá trị hẹp  :", r3, r4)

# cách 2 chỉ thử trên cây lệch 2000 nút để thấy O(n^2)
small = skew_chain(list(range(1, 2001)))
t0 = time.perf_counter()
assert longest_each_node(small) == 2000
print(f"cách 2 trên chuỗi 2000 nút: {(time.perf_counter() - t0)*1000:.0f} ms")

cây lệch tăng: đáp án 30000  (DFS 18 ms, BFS ngược 16 ms)


cây lệch zigzag: đáp án 2  (DFS 63 ms, BFS ngược 21 ms)
cây lệch giảm: đáp án 30000  (DFS 23 ms, BFS ngược 26 ms)


cây ngẫu nhiên 30000 nút, giá trị rộng: 2 2


cây ngẫu nhiên 30000 nút, giá trị hẹp  : 3 3


cách 2 trên chuỗi 2000 nút: 444 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Ghép hai nhánh mà không kiểm tra chiều

**Code sai** (ý nghĩ: "mỗi con cách cha đúng 1 thì nối được cả hai"):

```python
a = l + 1 if node.left  and abs(node.left.val  - node.val) == 1 else 1
b = r + 1 if node.right and abs(node.right.val - node.val) == 1 else 1
ans = max(ans, a + b - 1)
```

**Vì sao sai:** cây `[2, 1, 1]` có hai con đều hơn/kém `2` đúng 1, nhưng `1, 2, 1` **không đơn điệu**. Tăng rồi giảm không phải đường liên tiếp. Phải tách riêng chuỗi `+1` và chuỗi `−1`.

In [26]:
print("sai :", wrong_merge(build_tree([2, 1, 1])))      # 3
print("đúng:", longest_dp(build_tree([2, 1, 1])))       # 2

sai : 3
đúng: 2


### Lỗi 2: Quên trừ 1 khi ghép (đếm nút đỉnh hai lần)

**Code sai:**

```python
ans = max(ans, inc + dec)       # thiếu - 1
```

**Vì sao sai:** nút `v` thuộc **cả** chuỗi tăng lẫn chuỗi giảm, nên `inc + dec` đếm nó hai lần. Với một nút lẻ loi `inc = dec = 1`, kết quả sai thành `2` thay vì `1`.

In [27]:
def wrong_no_minus_one(root):
    ans = 0
    def dfs(node):
        nonlocal ans
        if node is None:
            return 0, 0
        inc = dec = 1
        for ch in (node.left, node.right):
            if ch is None:
                continue
            ci, cd = dfs(ch)
            if ch.val == node.val + 1:
                inc = max(inc, ci + 1)
            elif ch.val == node.val - 1:
                dec = max(dec, cd + 1)
        ans = max(ans, inc + dec)           # SAI: thiếu - 1
        return inc, dec
    dfs(root)
    return ans

print("sai :", wrong_no_minus_one(build_tree([7])))      # 2
print("đúng:", longest_dp(build_tree([7])))              # 1

sai : 2
đúng: 1


### Lỗi 3: Trả độ dài **đã uốn** lên cho nút cha

**Code sai:**

```python
ans = max(ans, inc + dec - 1)
return inc + dec - 1, ...          # SAI: trả đường đã uốn
```

**Vì sao sai:** cha chỉ có thể nối với một chuỗi **đi xuống một hướng** từ `v`. Đường đã uốn qua `v` có hai đầu rẽ vào hai con; nếu nối tiếp với cha, ta tạo ra đường có ba ngả (không phải đường đơn). Ví dụ tổng quát: chỉ trả `max(inc, dec)` hoặc nhầm hai thứ cũng gây sai.

Dưới đây là một bản sai trả `max(inc, dec)` lên cha (mất thông tin về chiều): cha không còn biết con đang cầm chuỗi tăng hay chuỗi giảm.

In [28]:
def wrong_return_max(root):
    ans = 0
    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        inc = dec = 1
        for ch in (node.left, node.right):
            if ch is None:
                continue
            c = dfs(ch)                       # chỉ nhận MỘT số, mất chiều
            if abs(ch.val - node.val) == 1:
                if ch.val == node.val + 1:
                    inc = max(inc, c + 1)
                else:
                    dec = max(dec, c + 1)
        ans = max(ans, inc + dec - 1)
        return max(inc, dec)                  # SAI: gộp hai chiều làm một
    dfs(root)
    return ans

print(wrong_return_max(build_tree([2, 1, 3])), longest_dp(build_tree([2, 1, 3])))     # có thể trùng đáp án đúng do may mắn

3 3


Ở cây trên, bản sai vẫn có thể ra đáp án đúng do may mắn; điều đó **không** chứng minh code đúng. Quy tắc nhớ: *nếu một hàm đệ quy cần báo cho cha cả "đi tăng" lẫn "đi giảm", hãy trả cả hai số riêng, đừng gộp.* Cách kiểm tra chắc chắn là dùng oracle (cách 1) với hàng nghìn cây ngẫu nhiên nhỏ, như ở mục 12 — hãy thử thay `longest_dp` bằng `wrong_return_max` ở đó để thấy nó bị bắt.

In [29]:
rng = random.Random(7)
bad = None
for _ in range(3000):
    vals = random_tree_list(rng, rng.randint(1, 10), 1, 4)
    root = build_tree(vals)
    if wrong_return_max(root) != longest_pairs(root):
        bad = vals
        break
print("cây đầu tiên bắt được cách sai:", bad)
if bad:
    print("sai:", wrong_return_max(build_tree(bad)), "| đúng:", longest_pairs(build_tree(bad)))

cây đầu tiên bắt được cách sai: [4, 3, 1, 4, 1, None, 3, None, None, None, None, None, 4, 1, 1, 3]
sai: 3 | đúng: 2


### Lỗi 4: Chỉ xét chiều tăng (nhầm sang bài 298)

**Code sai:**

```python
if ch.val == node.val + 1:
    inc = max(inc, ci + 1)
# thiếu nhánh giảm dần
```

**Vì sao sai:** đề này cho phép đường **giảm** và đường **uốn**. Bài 298 (Longest Consecutive Sequence I) chỉ cho đường **tăng đi từ cha xuống con**. Cây lệch giảm dần `[4, None, 3, None, 2, None, 1]` có đáp án `4` nhưng cách chỉ-tăng ra `1`.

In [30]:
def wrong_only_inc(root):
    ans = 0
    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        inc = 1
        for ch in (node.left, node.right):
            if ch is None:
                continue
            c = dfs(ch)
            if ch.val == node.val + 1:
                inc = max(inc, c + 1)
        ans = max(ans, inc)
        return inc
    dfs(root)
    return ans

t4 = [4, None, 3, None, 2, None, 1]
print("sai :", wrong_only_inc(build_tree(t4)))      # 1
print("đúng:", longest_dp(build_tree(t4)))          # 4

sai : 1
đúng: 4


### Lỗi 5: Biến đếm toàn cục không được đặt lại giữa các lần gọi

**Code sai:**

```python
class Solution:
    ans = 0                          # thuộc tính của LỚP: dùng chung cho mọi lần gọi
    def longestConsecutive(self, root):
        ...
        self.ans = max(self.ans, ...)
        return self.ans
```

**Vì sao sai:** LeetCode thường dùng **một** thể hiện `Solution` cho nhiều test (hoặc thuộc tính lớp bị giữ lại), nên kết quả của test trước rò sang test sau. Hãy luôn khởi tạo `ans = 0` **bên trong** hàm chính (như code nộp) hoặc gán `self.ans = 0` ở đầu hàm.

In [31]:
class BadSolution:
    ans = 0                                       # dùng chung -> rò kết quả
    def longestConsecutive(self, root):
        def dfs(node):
            if node is None:
                return 0, 0
            inc = dec = 1
            for ch in (node.left, node.right):
                if ch is None:
                    continue
                ci, cd = dfs(ch)
                if ch.val == node.val + 1:
                    inc = max(inc, ci + 1)
                elif ch.val == node.val - 1:
                    dec = max(dec, cd + 1)
            BadSolution.ans = max(BadSolution.ans, inc + dec - 1)
            return inc, dec
        dfs(root)
        return BadSolution.ans

print(BadSolution().longestConsecutive(build_tree([2, 1, 3])))     # 3
print(BadSolution().longestConsecutive(build_tree([1])), "(đúng phải là 1)")   # vẫn 3
print(Solution().longestConsecutive(build_tree([1])))             # 1

3
3 (đúng phải là 1)
1


## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu "đường đi có thể uốn qua một nút" trong cây** (cực đại hoá độ dài/tổng của đường đi): thuộc họ **Tree DP với đáp án toàn cục**. Mỗi nút báo lên cha một giá trị **một phía**, đồng thời dùng hai phía để cập nhật đáp án toàn cục (cùng khuôn với *Diameter of Binary Tree*, *Binary Tree Maximum Path Sum*).
- **Hai trạng thái cho mỗi nút** khi đề có "tăng hoặc giảm": giữ hai số (`inc`, `dec`) thay vì một. Muốn giữ chiều thì phải trả đủ cả hai.
- **Đáp án nằm ở nơi "ghép hai nhánh"**: công thức `a + b − 1` (hoặc `a + b`, tuỳ cách đếm cạnh/nút). Luôn kiểm tra: nút giữa có bị đếm hai lần không?
- Khi nghi ngờ đáp án, viết **oracle vét cạn** (mọi cặp nút) và so trên hàng nghìn cây nhỏ ngẫu nhiên với giá trị hẹp (để dễ có các chuỗi trùng nhau) — lỗi chiều/biên lộ ra rất nhanh.
- Cây lệch tới `3 * 10^4`: nghĩ tới `sys.setrecursionlimit` hoặc bản BFS-ngược.
- Chuyển từ bài 298 sang bài này: đổi "một chuỗi tăng đi xuống" thành "hai chuỗi (tăng, giảm) từ cùng một đỉnh".

## 15. Biến thể

1. **Bài 298 — Longest Consecutive Sequence I:** chỉ chuỗi **tăng** đi từ cha xuống con, không uốn. Chỉ cần `inc` một chiều, `ans = max(inc)`.
2. **Trả về chính đường đi** (danh sách giá trị) thay vì độ dài: giữ thêm con trỏ tới con kéo dài chuỗi để dựng lại.
3. **Bước tuỳ ý `k`** thay vì 1: thay `+1`, `−1` bằng `+k`, `−k` (đường có hiệu cố định `±k`).
4. **Cây N-phân:** vòng lặp qua danh sách con thay vì `(left, right)`; đỉnh `v` ghép **hai** chuỗi tốt nhất khác chiều (vẫn luôn khác con vì chiều khác nhau).
5. **Mảng thay cho cây — Longest Consecutive Sequence (128):** dãy các giá trị liên tiếp (không cần kề nhau trong mảng), giải bằng `set`.
6. **Đếm số đường đi liên tiếp** (chứ không chỉ độ dài lớn nhất): giữ thêm "số cách" cho mỗi `(inc, dec)`.

Dưới đây là biến thể 3 (bước `k`) — chỉ đổi `+1` thành `+k`:

In [32]:
def longest_step_k(root, k):
    ans = 0
    def dfs(node):
        nonlocal ans
        if node is None:
            return 0, 0
        inc = dec = 1
        for ch in (node.left, node.right):
            if ch is None:
                continue
            ci, cd = dfs(ch)
            if ch.val == node.val + k:
                inc = max(inc, ci + 1)
            elif ch.val == node.val - k:
                dec = max(dec, cd + 1)
        ans = max(ans, inc + dec - 1)
        return inc, dec
    dfs(root)
    return ans

t = build_tree([4, 2, 6, 0, None, None, 8])
print(longest_step_k(t, 2))     # 0 2 4 6 8 -> 5
print(longest_step_k(t, 1))     # không có hiệu 1 -> 1
assert longest_step_k(t, 1) == 1 and longest_step_k(t, 2) == 5
assert longest_step_k(build_tree([2, 1, 3, 0, None, None, 4]), 1) == longest_dp(build_tree([2, 1, 3, 0, None, None, 4]))

5
1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp nút | Dịch nguyên văn đề: dựng đường đi cho mỗi cặp rồi kiểm tra | `O(n³)` | `O(n)` | Chỉ làm oracle đối chứng với `n` nhỏ |
| 2. Mỗi nút làm đỉnh | Gom đường đi theo đỉnh, mỗi đỉnh tự dò hai chuỗi | `O(n²)` xấu nhất | `O(n)` | Bước trung gian để hiểu phân rã theo đỉnh; cây ngẫu nhiên nhỏ |
| 3. DFS `(inc, dec)` | Tree DP hậu tự, trả hai số một phía | `O(n)` | `O(h)` | **Cách nộp chuẩn**, ngắn và dễ nhớ |
| 4. BFS ngược | Cùng công thức, duyệt tầng rồi đọc ngược để không đệ quy | `O(n)` | `O(n)` | Cây lệch rất sâu, môi trường giới hạn đệ quy chặt |

Kết luận: nộp **cách 3**; nếu gặp `RecursionError` thì chuyển sang **cách 4**. Cách 1 giữ lại làm công cụ kiểm thử.

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có một cell để bạn làm. Các cell TODO bên dưới **chỉ khai báo** hàm rỗng hoặc gán `None`; hãy tự điền rồi bỏ comment dòng kiểm tra. Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Đoán kết quả trước khi chạy (viết ra giấy), rồi chạy để so sánh. Chú ý `is_consecutive` và `build_tree`.

In [33]:
print(is_consecutive([4, 3, 2, 1]))
print(is_consecutive([1, 2, 4, 3]))
print(tree_to_list(build_tree([1, None, 2, None, 3])))
print(longest_dp(build_tree([1, 2, 3])))
print(longest_dp(build_tree([3, 2, 4])))

True
False
[1, None, 2, None, 3]
2
3


### Bài 2 (tính tay)

Cho cây `[5, 6, 4, 7, None, None, 3]`:

```
        5
       / \
      6   4
     /     \
    7       3
```

Điền bảng `(inc, dec)` cho từng nút `7`, `6`, `3`, `4`, `5` rồi tìm đáp án. Gán kết quả vào biến `my_table` (dict từ giá trị nút sang tuple `(inc, dec)`) và `my_answer`. Hiện để `None`.

In [34]:
my_table = None      # ví dụ: {7: (?, ?), 6: (?, ?), 3: (?, ?), 4: (?, ?), 5: (?, ?)}
my_answer = None
# Sau khi điền, bỏ comment để kiểm tra:
# t = build_tree([5, 6, 4, 7, None, None, 3])
# truth = {n.val: p for n, p in all_inc_dec(t).items()}
# assert my_table == truth and my_answer == longest_dp(t)

### Bài 3 (điền TODO)

Viết `combine(v, children)` nhận giá trị nút `v` và danh sách các bộ ba `(giá_trị_con, inc_con, dec_con)` của các con **có thật**; trả về cặp `(inc, dec)` của nút `v`.

In [35]:
def combine(v, children):
    # TODO: bắt đầu với inc = dec = 1, duyệt từng con và cập nhật theo công thức
    pass

# Kiểm tra (bỏ comment sau khi viết xong):
# assert combine(2, []) == (1, 1)
# assert combine(2, [(3, 2, 1)]) == (3, 1)
# assert combine(2, [(1, 1, 3), (3, 2, 1)]) == (3, 4)

### Bài 4 (tự viết lại)

Viết `longest_298(root)`: độ dài chuỗi **tăng dài nhất đi từ cha xuống con** (bài 298: mỗi con hơn cha đúng `1`, không uốn, không đi giảm). Không xem lại code ở trên.

In [36]:
def longest_298(root):
    # TODO
    pass

# assert longest_298(build_tree([1, None, 3, 2, 4, None, None, None, 5])) == 3
# assert longest_298(build_tree([2, None, 3, 2, None, 1])) == 2
# assert longest_298(build_tree([4, None, 3, None, 2, None, 1])) == 1

### Bài 5 (biến thể / mini challenge)

Viết `longest_path_values(root)` trả về **chính một đường đi** (list giá trị, theo thứ tự đi dọc đường) có độ dài lớn nhất. Vì có thể có nhiều đường cùng dài nhất, hãy kiểm tra bằng hàm `valid_longest_path(root, path)`: đường phải liên tiếp (`is_consecutive`), có độ dài đúng bằng đáp án, và **thật sự** là một đường đi trong cây (bạn có thể dùng `path_between` hoặc tự nghĩ cách kiểm tra).

In [37]:
def valid_longest_path(root, path):
    if path is None or not is_consecutive(path) or len(path) != longest_dp(root):
        return False
    par, depth, order = parent_map(root)
    for u in order:                                   # tồn tại cặp (u, v) có đường đi đúng bằng path (hoặc đảo)
        for v in order:
            vals = path_between(u, v, par, depth)
            if vals == path:
                return True
    return False

def longest_path_values(root):
    # TODO: gợi ý - lưu thêm "con nào kéo dài inc/dec" để dựng lại đường
    return None

# for vals, expected in examples:
#     root = build_tree(vals)
#     path = longest_path_values(root)
#     assert valid_longest_path(root, path), (vals, path)

## 18. Gợi ý (3 mức)

**Bài 1.**
- Mức 1: `is_consecutive` kiểm tra hiệu giữa các phần tử kề nhau có luôn bằng một số cố định `±1` không.
- Mức 2: `build_tree([1, None, 2, None, 3])` là chuỗi lệch phải `1 → 2 → 3`; `tree_to_list` cắt `None` thừa ở cuối.
- Mức 3: kết quả theo thứ tự: `True`, `False`, `[1, None, 2, None, 3]`, `2`, `3`.

**Bài 2.**
- Mức 1: lá luôn có `(1, 1)`.
- Mức 2: nút `6` có con `7 = 6 + 1` nên `inc[6] = inc[7] + 1`; nút `4` có con `3 = 4 − 1` nên `dec[4] = dec[3] + 1`.
- Mức 3: ở gốc `5`, con `6` nối chuỗi tăng, con `4` nối chuỗi giảm: `inc = 3`, `dec = 3`, đáp án `3 + 3 − 1`.

**Bài 3.**
- Mức 1: vòng `for` qua `children`, mỗi phần tử là `(cv, ci, cd)`.
- Mức 2: nếu `cv == v + 1` thì cập nhật `inc`; nếu `cv == v − 1` thì cập nhật `dec` (dùng `elif`, vì không thể thoả cả hai).
- Mức 3: `inc = max(inc, ci + 1)` và `dec = max(dec, cd + 1)`; cuối cùng `return inc, dec`.

**Bài 4.**
- Mức 1: chỉ cần **một** giá trị trả về cho mỗi nút (chuỗi tăng dài nhất bắt đầu tại nút), không có `dec`.
- Mức 2: với mỗi con có `ch.val == node.val + 1`, `cur = max(cur, dfs(ch) + 1)`; đáp án là `max` của mọi `cur`.
- Mức 3: không có phép `− 1` vì không ghép hai nhánh.

**Bài 5.**
- Mức 1: cùng DP, nhưng lưu **đường** thay vì độ dài: `inc_path[v]` là list giá trị chuỗi `+1` dài nhất đi xuống từ `v`.
- Mức 2: đường uốn qua `v` = đảo ngược `dec_path` rồi nối `inc_path` (bỏ trùng nút `v`): `dec_path[::-1] + inc_path[1:]`.
- Mức 3: ở mỗi nút, so `len(dec_path) + len(inc_path) − 1` với độ dài tốt nhất đã thấy và lưu đường tương ứng.

## 19. Đáp án tham khảo

Các đáp án dưới đây đều chạy kiểm tra bằng `assert`.

In [38]:
# Bài 1
assert is_consecutive([4, 3, 2, 1]) is True
assert is_consecutive([1, 2, 4, 3]) is False
assert tree_to_list(build_tree([1, None, 2, None, 3])) == [1, None, 2, None, 3]
assert longest_dp(build_tree([1, 2, 3])) == 2
assert longest_dp(build_tree([3, 2, 4])) == 3
print("Bài 1: True, False, [1, None, 2, None, 3], 2, 3")

# Bài 2
t = build_tree([5, 6, 4, 7, None, None, 3])
ans_table = {7: (1, 1), 6: (2, 1), 3: (1, 1), 4: (1, 2), 5: (3, 3)}
truth = {n.val: p for n, p in all_inc_dec(t).items()}
assert truth == ans_table
assert longest_dp(t) == 5
print("Bài 2:", ans_table, "-> đáp án 5 (đường 7-6-5-4-3)")

Bài 1: True, False, [1, None, 2, None, 3], 2, 3
Bài 2: {7: (1, 1), 6: (2, 1), 3: (1, 1), 4: (1, 2), 5: (3, 3)} -> đáp án 5 (đường 7-6-5-4-3)


In [39]:
# Bài 3
def combine(v, children):
    inc = dec = 1
    for cv, ci, cd in children:
        if cv == v + 1:
            inc = max(inc, ci + 1)
        elif cv == v - 1:
            dec = max(dec, cd + 1)
    return inc, dec

assert combine(2, []) == (1, 1)
assert combine(2, [(3, 2, 1)]) == (3, 1)
assert combine(2, [(1, 1, 3), (3, 2, 1)]) == (3, 4)
assert combine(2, [(9, 5, 5)]) == (1, 1)
print("Bài 3: OK")

# Bài 4
def longest_298(root):
    best = 0
    def dfs(node):
        nonlocal best
        if node is None:
            return 0
        cur = 1
        for ch in (node.left, node.right):
            if ch is None:
                continue
            c = dfs(ch)
            if ch.val == node.val + 1:
                cur = max(cur, c + 1)
        best = max(best, cur)
        return cur
    dfs(root)
    return best

assert longest_298(build_tree([1, None, 3, 2, 4, None, None, None, 5])) == 3
assert longest_298(build_tree([2, None, 3, 2, None, 1])) == 2
assert longest_298(build_tree([4, None, 3, None, 2, None, 1])) == 1
print("Bài 4: OK")

Bài 3: OK
Bài 4: OK


In [40]:
# Bài 5
def longest_path_values(root):
    best = []
    def dfs(node):
        nonlocal best
        if node is None:
            return [], []
        inc_path, dec_path = [node.val], [node.val]
        for ch in (node.left, node.right):
            if ch is None:
                continue
            ci, cd = dfs(ch)
            if ch.val == node.val + 1 and len(ci) + 1 > len(inc_path):
                inc_path = [node.val] + ci
            elif ch.val == node.val - 1 and len(cd) + 1 > len(dec_path):
                dec_path = [node.val] + cd
        cand = dec_path[::-1] + inc_path[1:]          # ... v-2, v-1, v, v+1, v+2 ...
        if len(cand) > len(best):
            best = cand
        return inc_path, dec_path
    dfs(root)
    return best

for vals, expected in examples:
    root = build_tree(vals)
    path = longest_path_values(root)
    assert valid_longest_path(root, path), (vals, path)
    print(vals, "->", path)

rng = random.Random(99)
for _ in range(300):
    vals = random_tree_list(rng, rng.randint(1, 9), 1, 4)
    root = build_tree(vals)
    assert valid_longest_path(root, longest_path_values(root)), vals
print("Bài 5: OK (đã kiểm tra 300 cây ngẫu nhiên)")

[1, 2, 3] -> [1, 2]
[2, 1, 3] -> [1, 2, 3]
[2, 1, 1] -> [1, 2]
[2, 1, 3, 0, None, None, 4] -> [0, 1, 2, 3, 4]
Bài 5: OK (đã kiểm tra 300 cây ngẫu nhiên)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Tree DP "đường đi uốn qua nút, hai chiều":**

```python
class Solution:
    def longestConsecutive(self, root: Optional[TreeNode]) -> int:
        ans = 0

        def dfs(node):
            nonlocal ans
            if node is None:
                return 0, 0
            inc = dec = 1                              # chính node là chuỗi dài 1
            for ch in (node.left, node.right):
                if ch is None:
                    continue
                ci, cd = dfs(ch)                       # gọi cho CẢ HAI con
                if ch.val == node.val + 1:
                    inc = max(inc, ci + 1)
                elif ch.val == node.val - 1:
                    dec = max(dec, cd + 1)
            ans = max(ans, inc + dec - 1)              # ghép hai phía, trừ 1 vì đếm node hai lần
            return inc, dec                            # chỉ trả MỘT phía mỗi chiều

        dfs(root)
        return ans
```

**Checklist trước khi nộp:**

- [ ] Gặp `None` trả `(0, 0)`; lá cho `(1, 1)`.
- [ ] `inc`, `dec` khởi tạo `1` (không phải `0`).
- [ ] Gọi `dfs` cho cả hai con **trước** khi so giá trị (không bỏ sót con nào).
- [ ] Dùng `elif` giữa nhánh `+1` và `−1` (một con chỉ thoả một trong hai).
- [ ] Cập nhật đáp án bằng `inc + dec - 1` (có trừ 1).
- [ ] Trả về **hai số một phía**, không trả đường đã uốn.
- [ ] `ans` khởi tạo trong hàm chính, không là thuộc tính lớp dùng chung.
- [ ] Cây lệch 30 000 nút: cân nhắc cách 4 (không đệ quy) nếu môi trường giới hạn đệ quy.
- [ ] Đã thử: cây một nút, `[2,1,1]` (không đơn điệu), chuỗi giảm, chuỗi tăng, đường uốn qua gốc.